# 05 Estimate Watch Time From Enriched Watch Events

This notebook estimates watch time from the enriched event table created in notebook 04.

The estimate follows the core logic of the original watch-time script: combine each video's metadata duration with the start time of the next observed watch event for the same participant. If the next watch starts before the current video's metadata duration has elapsed, the unelapsed duration is treated as skipped. If the next watch starts after the video's expected end, the full video duration is counted.

This is still an estimate. YouTube Takeout watch history gives start times, not completion times. The calculation is most useful for aggregate summaries across many events, and the assumptions should be reviewed before using it for substantive claims.

## What This Notebook Produces

The notebook reads `outputs/tables/video_histories_enriched.csv` and writes three files:

- `outputs/tables/video_histories_watchtime.csv`: one row per watch event, with watch-time helper columns added.
- `outputs/tables/watchtime_summary.csv`: one row per participant with all-data watch-time totals by Shorts/Longs and day/night.
- `outputs/tables/watchtime_diagnostics.csv`: compact diagnostics and assumption settings.

Unlike the legacy script, this teaching version does not create separate one-month, six-month, or one-year windows. It runs across all events in the enriched mock dataset.

## Locate Project Files

Paths are kept relative to the Quarto project root so the notebook can be rendered locally or as part of the website without exposing private machine paths.

In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        has_quarto_project = (candidate / "_quarto.yml").exists()
        has_outputs = (candidate / "outputs" / "tables").exists()
        if has_quarto_project and has_outputs:
            return candidate
    raise FileNotFoundError(
        "Could not find the Quarto project root. Run this notebook from "
        "inside the public YouTube donation method repository."
    )


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

ENRICHED_VIDEO_HISTORIES_PATH = OUTPUT_DIR / "video_histories_enriched.csv"
WATCHTIME_EVENTS_PATH = OUTPUT_DIR / "video_histories_watchtime.csv"
WATCHTIME_SUMMARY_PATH = OUTPUT_DIR / "watchtime_summary.csv"
WATCHTIME_DIAGNOSTICS_PATH = OUTPUT_DIR / "watchtime_diagnostics.csv"

print(f"Project folder: {PROJECT_ROOT.name}")
print(f"Input table: {ENRICHED_VIDEO_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Output folder: {OUTPUT_DIR.relative_to(PROJECT_ROOT).as_posix()}")

## Configure Assumptions

The key assumption is how to handle the last observed watch event for each participant. There is no next watch event that can reveal whether the participant stopped early. The conservative default is therefore to leave the last event's estimated watch time unknown.

Set `ASSUME_FULL_DURATION_FOR_LAST_EVENT = True` if a project wants to count the full metadata duration for each participant's last observed event. This is more complete but also stronger.

In [ ]:
ASSUME_FULL_DURATION_FOR_LAST_EVENT = False
CAP_ESTIMATED_WATCHTIME_TO_DURATION = True

print("Assume full duration for last observed event:", ASSUME_FULL_DURATION_FOR_LAST_EVENT)
print("Cap estimated watch time to metadata duration:", CAP_ESTIMATED_WATCHTIME_TO_DURATION)

## Load The Enriched Watch Table

Notebook 04 already joined watch events to metadata and created the day/night and Shorts/Longs labels. This notebook reuses those fields instead of recomputing timezone-sensitive classifications.

In [ ]:
if not ENRICHED_VIDEO_HISTORIES_PATH.exists():
    raise FileNotFoundError(
        f"Missing input table: {ENRICHED_VIDEO_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}"
    )

video_histories_enriched = pd.read_csv(
    ENRICHED_VIDEO_HISTORIES_PATH,
    dtype={"Participant ID": "string", "video_id": "string"},
    low_memory=False,
)

required_columns = {
    "Participant ID",
    "time",
    "video_id",
    "duration_seconds",
    "is_short",
    "is_long",
    "time_of_day",
    "watch_local_time",
}
missing_columns = sorted(required_columns - set(video_histories_enriched.columns))
if missing_columns:
    raise ValueError(
        f"Missing required columns in video_histories_enriched.csv: {missing_columns}"
    )

video_histories_enriched["time"] = pd.to_datetime(
    video_histories_enriched["time"], utc=True, format="mixed", errors="coerce"
)
if video_histories_enriched["time"].isna().any():
    raise ValueError("Some watch events have invalid UTC timestamps.")

video_histories_enriched["duration_seconds"] = pd.to_numeric(
    video_histories_enriched["duration_seconds"], errors="coerce"
)

print(f"Loaded {len(video_histories_enriched)} enriched watch events")
print(f"Participants: {video_histories_enriched['Participant ID'].nunique()}")
print(f"Events with metadata duration: {video_histories_enriched['duration_seconds'].notna().sum()}")

video_histories_enriched.head(5)

## Estimate Event-Level Watch Time

For each participant, the events are sorted by watch start time. The next watch start is then used as an upper bound for how long the current event could have been watched.

For rows with a known metadata duration and a known next watch start:

- if the next watch starts before the current video would have ended, estimated watch time is the time until the next watch;
- if the next watch starts after the current video would have ended, estimated watch time is the full metadata duration.

Rows without metadata duration remain unknown. Last observed events remain unknown by default, unless configured otherwise above.

In [ ]:
video_histories_watchtime = video_histories_enriched.copy()
video_histories_watchtime["watch_event_position"] = range(len(video_histories_watchtime))

video_histories_watchtime = video_histories_watchtime.sort_values(
    ["Participant ID", "time", "watch_event_position"]
).reset_index(drop=True)

video_histories_watchtime["participant_event_number"] = (
    video_histories_watchtime.groupby("Participant ID").cumcount() + 1
)

video_histories_watchtime["next_watch_time"] = (
    video_histories_watchtime.groupby("Participant ID")["time"].shift(-1)
)

video_histories_watchtime["seconds_to_next_watch"] = (
    video_histories_watchtime["next_watch_time"] - video_histories_watchtime["time"]
).dt.total_seconds()

video_histories_watchtime["expected_end_time"] = (
    video_histories_watchtime["time"]
    + pd.to_timedelta(video_histories_watchtime["duration_seconds"], unit="s")
)

has_duration = video_histories_watchtime["duration_seconds"].notna()
has_next_watch = video_histories_watchtime["next_watch_time"].notna()
valid_duration = has_duration & video_histories_watchtime["duration_seconds"].ge(0)
valid_next_gap = has_next_watch & video_histories_watchtime["seconds_to_next_watch"].ge(0)

video_histories_watchtime["estimated_watch_seconds"] = pd.NA

bounded_rows = valid_duration & valid_next_gap
video_histories_watchtime.loc[bounded_rows, "estimated_watch_seconds"] = video_histories_watchtime.loc[
    bounded_rows, ["duration_seconds", "seconds_to_next_watch"]
].min(axis=1)

if ASSUME_FULL_DURATION_FOR_LAST_EVENT:
    last_rows_with_duration = valid_duration & ~has_next_watch
    video_histories_watchtime.loc[last_rows_with_duration, "estimated_watch_seconds"] = (
        video_histories_watchtime.loc[last_rows_with_duration, "duration_seconds"]
    )

video_histories_watchtime["estimated_watch_seconds"] = pd.to_numeric(
    video_histories_watchtime["estimated_watch_seconds"], errors="coerce"
)

if CAP_ESTIMATED_WATCHTIME_TO_DURATION:
    video_histories_watchtime["estimated_watch_seconds"] = video_histories_watchtime[
        ["estimated_watch_seconds", "duration_seconds"]
    ].min(axis=1)

video_histories_watchtime["estimated_watch_seconds"] = (
    video_histories_watchtime["estimated_watch_seconds"].clip(lower=0)
)

video_histories_watchtime["overlap_with_next_seconds"] = (
    video_histories_watchtime["duration_seconds"] - video_histories_watchtime["seconds_to_next_watch"]
).clip(lower=0)
video_histories_watchtime.loc[~bounded_rows, "overlap_with_next_seconds"] = pd.NA

video_histories_watchtime["estimated_watch_minutes"] = (
    video_histories_watchtime["estimated_watch_seconds"] / 60
)
video_histories_watchtime["estimated_watch_hours"] = (
    video_histories_watchtime["estimated_watch_seconds"] / 3600
)

video_histories_watchtime["watchtime_status"] = "estimated"
video_histories_watchtime.loc[~has_duration, "watchtime_status"] = "missing_duration"
video_histories_watchtime.loc[has_duration & video_histories_watchtime["duration_seconds"].lt(0), "watchtime_status"] = "invalid_duration"
video_histories_watchtime.loc[valid_duration & has_next_watch & video_histories_watchtime["seconds_to_next_watch"].lt(0), "watchtime_status"] = "invalid_next_watch_order"
video_histories_watchtime.loc[valid_duration & ~has_next_watch, "watchtime_status"] = (
    "assumed_full_duration_last_event" if ASSUME_FULL_DURATION_FOR_LAST_EVENT else "last_observed_event_unknown"
)
video_histories_watchtime.loc[
    bounded_rows & video_histories_watchtime["seconds_to_next_watch"].lt(video_histories_watchtime["duration_seconds"]),
    "watchtime_status",
] = "cut_off_by_next_watch"
video_histories_watchtime.loc[
    bounded_rows & video_histories_watchtime["seconds_to_next_watch"].ge(video_histories_watchtime["duration_seconds"]),
    "watchtime_status",
] = "full_duration_before_next_watch"

video_histories_watchtime[[
    "Participant ID",
    "time",
    "video_id",
    "duration_seconds",
    "next_watch_time",
    "seconds_to_next_watch",
    "estimated_watch_seconds",
    "watchtime_status",
]].head(10)

## Build All-Data Participant Summaries

The legacy script exported one-month, six-month, and one-year summaries. This notebook keeps the same conceptual breakdown by participant, day/night, and Shorts/Longs, but runs it across the full mock dataset only.

The summary reports total estimated hours, not average daily hours, because no period window is imposed here.

In [ ]:
summary_events = video_histories_watchtime.copy()
summary_events["video_format"] = "long"
summary_events.loc[summary_events["is_short"].astype("string").str.lower().eq("true"), "video_format"] = "short"
summary_events["has_estimated_watchtime"] = summary_events["estimated_watch_seconds"].notna()

watchtime_by_group = (
    summary_events
    .groupby(["Participant ID", "video_format", "time_of_day"], dropna=False)["estimated_watch_seconds"]
    .sum(min_count=1)
    .reset_index()
)
watchtime_by_group["summary_column"] = (
    "watchtime_"
    + watchtime_by_group["video_format"].astype(str)
    + "_"
    + watchtime_by_group["time_of_day"].astype(str)
    + "_hours"
)
watchtime_wide = (
    watchtime_by_group
    .pivot(index="Participant ID", columns="summary_column", values="estimated_watch_seconds")
    .reset_index()
)

for column in watchtime_wide.columns:
    if column != "Participant ID":
        watchtime_wide[column] = watchtime_wide[column] / 3600

base_summary = (
    summary_events
    .groupby("Participant ID")
    .agg(
        watch_events=("video_id", "size"),
        events_with_estimated_watchtime=("has_estimated_watchtime", "sum"),
        total_estimated_watch_seconds=("estimated_watch_seconds", "sum"),
        missing_duration_events=("watchtime_status", lambda s: (s == "missing_duration").sum()),
        last_observed_unknown_events=("watchtime_status", lambda s: (s == "last_observed_event_unknown").sum()),
        cut_off_by_next_watch_events=("watchtime_status", lambda s: (s == "cut_off_by_next_watch").sum()),
        full_duration_before_next_watch_events=("watchtime_status", lambda s: (s == "full_duration_before_next_watch").sum()),
    )
    .reset_index()
)
base_summary["total_estimated_watch_hours"] = base_summary["total_estimated_watch_seconds"] / 3600

watchtime_summary = base_summary.merge(watchtime_wide, on="Participant ID", how="left").fillna(0)

preferred_summary_columns = [
    "Participant ID",
    "watch_events",
    "events_with_estimated_watchtime",
    "missing_duration_events",
    "last_observed_unknown_events",
    "cut_off_by_next_watch_events",
    "full_duration_before_next_watch_events",
    "total_estimated_watch_seconds",
    "total_estimated_watch_hours",
    "watchtime_short_day_hours",
    "watchtime_short_night_hours",
    "watchtime_long_day_hours",
    "watchtime_long_night_hours",
]
for column in preferred_summary_columns:
    if column not in watchtime_summary.columns:
        watchtime_summary[column] = 0

watchtime_summary = watchtime_summary[preferred_summary_columns]

watchtime_summary

## Diagnostics

These diagnostics make the estimation assumptions and missingness visible. In particular, missing duration and last observed events limit how much watch time can be estimated from the available data.

In [ ]:
def count_status(status):
    return int((video_histories_watchtime["watchtime_status"] == status).sum())

watchtime_diagnostics = pd.DataFrame(
    [
        {"section": "inputs", "measure": "watch_events", "value": len(video_histories_watchtime)},
        {"section": "inputs", "measure": "participants", "value": video_histories_watchtime["Participant ID"].nunique()},
        {"section": "inputs", "measure": "events_with_duration", "value": int(video_histories_watchtime["duration_seconds"].notna().sum())},
        {"section": "inputs", "measure": "events_missing_duration", "value": int(video_histories_watchtime["duration_seconds"].isna().sum())},
        {"section": "sequence", "measure": "events_with_next_watch", "value": int(video_histories_watchtime["next_watch_time"].notna().sum())},
        {"section": "sequence", "measure": "last_observed_events", "value": int(video_histories_watchtime["next_watch_time"].isna().sum())},
        {"section": "watchtime", "measure": "events_with_estimated_watchtime", "value": int(video_histories_watchtime["estimated_watch_seconds"].notna().sum())},
        {"section": "watchtime", "measure": "total_estimated_watch_hours", "value": round(float(video_histories_watchtime["estimated_watch_hours"].sum(skipna=True)), 3)},
        {"section": "status", "measure": "cut_off_by_next_watch", "value": count_status("cut_off_by_next_watch")},
        {"section": "status", "measure": "full_duration_before_next_watch", "value": count_status("full_duration_before_next_watch")},
        {"section": "status", "measure": "missing_duration", "value": count_status("missing_duration")},
        {"section": "status", "measure": "last_observed_event_unknown", "value": count_status("last_observed_event_unknown")},
        {"section": "settings", "measure": "assume_full_duration_for_last_event", "value": str(ASSUME_FULL_DURATION_FOR_LAST_EVENT)},
        {"section": "settings", "measure": "cap_estimated_watchtime_to_duration", "value": str(CAP_ESTIMATED_WATCHTIME_TO_DURATION)},
    ]
)

watchtime_diagnostics

## Validate And Save Outputs

The event-level output should keep exactly the same number of rows as `video_histories_enriched.csv`. Estimated watch time, when present, should be non-negative and no larger than the metadata duration.

In [ ]:
if len(video_histories_watchtime) != len(video_histories_enriched):
    raise ValueError("Watch-time output changed the number of watch events.")

estimated_rows = video_histories_watchtime["estimated_watch_seconds"].notna()
if (video_histories_watchtime.loc[estimated_rows, "estimated_watch_seconds"] < 0).any():
    raise ValueError("Estimated watch time contains negative values.")

if CAP_ESTIMATED_WATCHTIME_TO_DURATION:
    too_large = (
        video_histories_watchtime.loc[estimated_rows, "estimated_watch_seconds"]
        > video_histories_watchtime.loc[estimated_rows, "duration_seconds"]
    )
    if too_large.any():
        raise ValueError("Estimated watch time exceeds metadata duration for some rows.")

video_histories_watchtime.to_csv(WATCHTIME_EVENTS_PATH, index=False)
watchtime_summary.to_csv(WATCHTIME_SUMMARY_PATH, index=False)
watchtime_diagnostics.to_csv(WATCHTIME_DIAGNOSTICS_PATH, index=False)

print(f"Saved {WATCHTIME_EVENTS_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Saved {WATCHTIME_SUMMARY_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Saved {WATCHTIME_DIAGNOSTICS_PATH.relative_to(PROJECT_ROOT).as_posix()}")

## Result

The event-level table keeps the watch-history unit of analysis and adds the watch-time estimate. The participant summary collapses those event estimates across the full mock dataset.

In [ ]:
print(f"video_histories_watchtime.csv ({len(video_histories_watchtime)} rows)")
display(video_histories_watchtime.head(5))

print(f"watchtime_summary.csv ({len(watchtime_summary)} rows)")
display(watchtime_summary.head(5))

print(f"watchtime_diagnostics.csv ({len(watchtime_diagnostics)} rows)")
display(watchtime_diagnostics.head(20))